In [ ]:
# %%
import os, math, time, json, random
from dataclasses import dataclass, asdict
from typing import Tuple, Dict, Optional, List

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

torch.backends.cudnn.benchmark = True

print("torch:", torch.__version__)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

def set_seed(seed: int = 17):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def ensure_dir(p: str):
    os.makedirs(p, exist_ok=True)

plt.rcParams["figure.dpi"] = 120


In [ ]:
# %%
@dataclass
class Config:
    H: int = 128
    W: int = 128
    M: int = 12
    p: float = 2.0

    n_train: int = 20000
    n_val: int = 5000
    n_test: int = 3000

    batch_size: int = 64
    epochs: int = 200
    lr: float = 1e-4
    weight_decay: float = 1e-2

    ddpm_T: int = 100
    ddpm_beta_start: float = 1e-4
    ddpm_beta_end: float = 2e-2

    fm_t_eps: float = 1e-5

    base_ch: int = 32
    down_levels: int = 3

    out_dir: str = "./runs_flowgen"
    run_name: str = "demo"

cfg = Config()
ensure_dir(cfg.out_dir)
print(asdict(cfg))


In [ ]:
# %%
def make_grid(H: int, W: int, L: float = 2*math.pi):
    x = np.linspace(0, L, W, endpoint=False)
    y = np.linspace(0, L, H, endpoint=False)
    return np.meshgrid(x, y, indexing="xy")

def sample_fourier_streamfunction(H: int, W: int, M: int, p: float, rng: np.random.Generator):
    X, Y = make_grid(H, W)
    psi = np.zeros((H, W), dtype=np.float32)
    for m in range(1, M+1):
        for n in range(1, M+1):
            k2 = m*m + n*n
            sigma = k2 ** (-p/2.0)
            a = rng.normal(0.0, sigma)
            b = rng.normal(0.0, sigma)
            psi += a * np.sin(m*X + n*Y) + b * np.cos(m*X - n*Y)
    return psi

def laplacian_periodic(f: np.ndarray, L: float = 2*math.pi):
    H, W = f.shape
    dx = L / W
    dy = L / H
    f_xx = (np.roll(f, -1, 1) - 2*f + np.roll(f, 1, 1)) / (dx*dx)
    f_yy = (np.roll(f, -1, 0) - 2*f + np.roll(f, 1, 0)) / (dy*dy)
    return (f_xx + f_yy).astype(np.float32)

def generate_omega_samples(N: int, H: int, W: int, M: int, p: float, seed: int):
    rng = np.random.default_rng(seed)
    out = np.zeros((N, H, W), dtype=np.float32)
    for i in range(N):
        psi = sample_fourier_streamfunction(H, W, M, p, rng)
        out[i] = -laplacian_periodic(psi)
        if (i+1) % 1000 == 0:
            print(f"generated {i+1}/{N}")
    return out


In [ ]:
# %%
set_seed(17)
preview = generate_omega_samples(8, cfg.H, cfg.W, cfg.M, cfg.p, seed=17)
vmin, vmax = np.percentile(preview, [2,98])
fig, axes = plt.subplots(2,4, figsize=(10,4))
for ax, om in zip(axes.ravel(), preview):
    im = ax.imshow(om, origin="lower", vmin=vmin, vmax=vmax)
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.7)
plt.suptitle("Preview omega samples (unnormalized)")
plt.show()


In [ ]:
# %%
set_seed(17)
omegas_train = generate_omega_samples(cfg.n_train, cfg.H, cfg.W, cfg.M, cfg.p, seed=17)
omegas_val   = generate_omega_samples(cfg.n_val,   cfg.H, cfg.W, cfg.M, cfg.p, seed=42)
omegas_test  = generate_omega_samples(cfg.n_test,  cfg.H, cfg.W, cfg.M, cfg.p, seed=123)

mu = omegas_train.mean()
std = omegas_train.std() + 1e-8

def standardize(x): 
    return ((x - mu) / std).astype(np.float32)

omegas_train_z = standardize(omegas_train)
omegas_val_z   = standardize(omegas_val)
omegas_test_z  = standardize(omegas_test)

print("train mean/std (after):", omegas_train_z.mean(), omegas_train_z.std())

with open(os.path.join(cfg.out_dir, cfg.run_name + "_data_stats.json"), "w") as f:
    json.dump({"mu": float(mu), "std": float(std)}, f, indent=2)


In [ ]:
# %%
class OmegaDataset(Dataset):
    def __init__(self, omegas_z: np.ndarray):
        self.x = torch.from_numpy(omegas_z).unsqueeze(1)  # (N,1,H,W)
    def __len__(self): 
        return self.x.shape[0]
    def __getitem__(self, i): 
        return self.x[i]

train_loader = DataLoader(OmegaDataset(omegas_train_z), batch_size=cfg.batch_size, shuffle=True, drop_last=True)
val_loader   = DataLoader(OmegaDataset(omegas_val_z),   batch_size=cfg.batch_size, shuffle=False)


In [ ]:
# %%
class FourierTimeEmbedding(nn.Module):
    def __init__(self, embed_dim: int = 128, max_freq: float = 10.0):
        super().__init__()
        assert embed_dim % 2 == 0
        self.register_buffer("freqs", torch.linspace(1.0, max_freq, embed_dim // 2))
    def forward(self, t: torch.Tensor):
        angles = t[:, None] * self.freqs[None, :] * 2 * math.pi
        return torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)

class ResBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, t_dim: int, groups: int = 8):
        super().__init__()
        self.norm1 = nn.GroupNorm(groups, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(groups, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.t_proj = nn.Linear(t_dim, out_ch)
        self.act = nn.SiLU()
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, t_emb):
        h = self.conv1(self.act(self.norm1(x)))
        h = h + self.t_proj(self.act(t_emb))[:, :, None, None]
        h = self.conv2(self.act(self.norm2(h)))
        return h + self.skip(x)

class UNetLite(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base_ch=32, down_levels=3, t_dim=128):
        super().__init__()
        self.t_embed = nn.Sequential(
            FourierTimeEmbedding(t_dim),
            nn.Linear(t_dim, t_dim), nn.SiLU(),
            nn.Linear(t_dim, t_dim),
        )
        chs = [base_ch * (2**i) for i in range(down_levels+1)]
        self.in_conv = nn.Conv2d(in_ch, chs[0], 3, padding=1)

        self.down_blocks = nn.ModuleList()
        self.downsample = nn.ModuleList()
        for i in range(down_levels):
            self.down_blocks.append(nn.ModuleList([
                ResBlock(chs[i], chs[i], t_dim),
                ResBlock(chs[i], chs[i+1], t_dim),
            ]))
            self.downsample.append(nn.Conv2d(chs[i+1], chs[i+1], 4, stride=2, padding=1))

        self.mid1 = ResBlock(chs[-1], chs[-1], t_dim)
        self.mid2 = ResBlock(chs[-1], chs[-1], t_dim)

        self.upsample = nn.ModuleList()
        self.up_blocks = nn.ModuleList()
        for i in reversed(range(down_levels)):
            self.upsample.append(nn.ConvTranspose2d(chs[i+1], chs[i+1], 4, stride=2, padding=1))
            self.up_blocks.append(nn.ModuleList([
                ResBlock(chs[i+1] + chs[i+1], chs[i+1], t_dim),
                ResBlock(chs[i+1], chs[i], t_dim),
            ]))

        self.out_norm = nn.GroupNorm(8, chs[0])
        self.out_conv = nn.Conv2d(chs[0], out_ch, 3, padding=1)

    def forward(self, x, t):
        t_emb = self.t_embed(t)
        h = self.in_conv(x)
        skips = []
        for blocks, down in zip(self.down_blocks, self.downsample):
            h = blocks[0](h, t_emb)
            h = blocks[1](h, t_emb)
            skips.append(h)
            h = down(h)
        h = self.mid1(h, t_emb)
        h = self.mid2(h, t_emb)
        for up, blocks in zip(self.upsample, self.up_blocks):
            h = up(h)
            skip = skips.pop()
            if h.shape[-2:] != skip.shape[-2:]:
                Hm = min(h.shape[-2], skip.shape[-2])
                Wm = min(h.shape[-1], skip.shape[-1])
                h = h[..., :Hm, :Wm]
                skip = skip[..., :Hm, :Wm]
            h = torch.cat([h, skip], dim=1)
            h = blocks[0](h, t_emb)
            h = blocks[1](h, t_emb)
        return self.out_conv(F.silu(self.out_norm(h)))

tmp = UNetLite(base_ch=cfg.base_ch, down_levels=cfg.down_levels).to(DEVICE)
print(tmp(torch.randn(2,1,cfg.H,cfg.W, device=DEVICE), torch.rand(2, device=DEVICE)).shape)


In [ ]:
# %%
class DDPM:
    def __init__(self, T: int, beta_start: float, beta_end: float, device: torch.device):
        self.T = T
        self.device = device

        betas = torch.linspace(beta_start, beta_end, T, device=device)
        alphas = 1.0 - betas
        alphas_bar = torch.cumprod(alphas, dim=0)

        self.betas = betas
        self.alphas = alphas
        self.alphas_bar = alphas_bar
        self.sqrt_alphas_bar = torch.sqrt(alphas_bar)
        self.sqrt_one_minus_alphas_bar = torch.sqrt(1.0 - alphas_bar)

    def q_sample(self, x0: torch.Tensor, t_idx: torch.Tensor, eps: torch.Tensor) -> torch.Tensor:
        a = self.sqrt_alphas_bar[t_idx][:, None, None, None]
        b = self.sqrt_one_minus_alphas_bar[t_idx][:, None, None, None]
        return a * x0 + b * eps

    @torch.no_grad()
    def p_sample(self, model, x_t: torch.Tensor, t_idx: int) -> torch.Tensor:
        B = x_t.shape[0]
        t = torch.full((B,), (t_idx + 1) / self.T, device=self.device)
        eps_theta = model(x_t, t)

        beta_t = self.betas[t_idx]
        alpha_t = self.alphas[t_idx]
        alpha_bar_t = self.alphas_bar[t_idx]

        coef1 = 1.0 / torch.sqrt(alpha_t)
        coef2 = beta_t / torch.sqrt(1.0 - alpha_bar_t)
        mean = coef1 * (x_t - coef2 * eps_theta)

        if t_idx == 0:
            return mean

        noise = torch.randn_like(x_t)
        return mean + torch.sqrt(beta_t) * noise

    @torch.no_grad()
    def sample_full(self, model, shape: Tuple[int,int,int,int]) -> torch.Tensor:
        """
        Strict full reverse chain: steps = T (no skipping).
        This is what we use for quantitative tradeoff evaluation.
        """
        x = torch.randn(shape, device=self.device)
        for t_idx in range(self.T - 1, -1, -1):
            x = self.p_sample(model, x, t_idx)
        return x

    @torch.no_grad()
    def sample_with_steps_for_viz(self, model, shape: Tuple[int,int,int,int], steps: int = 50) -> torch.Tensor:
        """
        For visualization only: allow strided indices to show intermediate states.
        Not used for strict quantitative comparison unless you switch to DDIM.
        """
        x = torch.randn(shape, device=self.device)

        if steps >= self.T:
            idxs = list(range(self.T - 1, -1, -1))
        else:
            stride = (self.T - 1) / (steps - 1)
            idxs = [int(round(i * stride)) for i in reversed(range(steps))]
            idxs = sorted(list(set(idxs)), reverse=True)
            if idxs[-1] != 0:
                idxs.append(0)

        for t_idx in idxs:
            x = self.p_sample(model, x, t_idx)
        return x


In [ ]:
# %%
def fm_sample_xt(x0: torch.Tensor, x1: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    return (1 - t)[:, None, None, None] * x0 + t[:, None, None, None] * x1

@torch.no_grad()
def fm_sample_ode(model, shape: Tuple[int,int,int,int], n_steps: int = 10, method: str = "euler", device=None) -> torch.Tensor:
    device = device or next(model.parameters()).device
    B = shape[0]
    x = torch.randn(shape, device=device)
    dt = 1.0 / n_steps

    def v(x, t_scalar: float):
        t = torch.full((B,), t_scalar, device=device)
        return model(x, t)

    for step in range(n_steps):
        t = step / n_steps
        if method == "euler":
            x = x + dt * v(x, t)
        elif method == "rk4":
            k1 = v(x, t)
            k2 = v(x + 0.5 * dt * k1, t + 0.5 * dt)
            k3 = v(x + 0.5 * dt * k2, t + 0.5 * dt)
            k4 = v(x + dt * k3, t + dt)
            x = x + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)
        else:
            raise ValueError("method must be 'euler' or 'rk4'")

    return x


In [ ]:
# %%
def radial_spectrum_2d(field: np.ndarray):
    H, W = field.shape
    Fk = np.fft.fft2(field)
    P = (np.abs(Fk) ** 2).astype(np.float64)

    ky = np.fft.fftfreq(H) * H
    kx = np.fft.fftfreq(W) * W
    KX, KY = np.meshgrid(kx, ky, indexing="xy")
    K = np.sqrt(KX**2 + KY**2)

    k_int = np.round(K).astype(np.int32)
    kmax = k_int.max()
    E = np.zeros(kmax + 1, dtype=np.float64)
    cnt = np.zeros(kmax + 1, dtype=np.int64)

    for kk in range(kmax + 1):
        mask = (k_int == kk)
        cnt[kk] = mask.sum()
        if cnt[kk] > 0:
            E[kk] = P[mask].mean()
        else:
            E[kk] = 0.0

    return np.arange(kmax + 1), E

def spec_err(gen: np.ndarray, real: np.ndarray, eps: float = 1e-12) -> float:
    _, Eg = radial_spectrum_2d(gen)
    _, Er = radial_spectrum_2d(real)
    L = min(len(Eg), len(Er))
    Eg = Eg[:L] + eps
    Er = Er[:L] + eps
    # drop k=0 to avoid DC dominating (more stable)
    return float(np.sum(np.abs(np.log(Eg[1:]) - np.log(Er[1:]))))

def grad_periodic(f: np.ndarray, L: float = 2*math.pi):
    H, W = f.shape
    dx = L / W
    dy = L / H
    fx = (np.roll(f, -1, axis=1) - np.roll(f, 1, axis=1)) / (2 * dx)
    fy = (np.roll(f, -1, axis=0) - np.roll(f, 1, axis=0)) / (2 * dy)
    return fx.astype(np.float32), fy.astype(np.float32)

def h1_error(gen: np.ndarray, real: np.ndarray) -> float:
    d = (gen - real).astype(np.float32)
    fx, fy = grad_periodic(d)
    val = (d*d + fx*fx + fy*fy).mean()
    return float(math.sqrt(val))

def batch_metrics(gen_batch: np.ndarray, real_batch: np.ndarray) -> Dict[str, float]:
    assert gen_batch.shape == real_batch.shape
    N = gen_batch.shape[0]
    se = 0.0
    h1 = 0.0
    for i in range(N):
        se += spec_err(gen_batch[i], real_batch[i])
        h1 += h1_error(gen_batch[i], real_batch[i])
    return {"SpecErr": se/N, "H1Err": h1/N}

def structure_complexity(field: np.ndarray) -> float:
    fx, fy = grad_periodic(field)
    return float(np.sqrt(np.mean(fx*fx + fy*fy)))


In [ ]:
# %%
def poisson_solve_streamfunction_from_vorticity(omega: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    H, W = omega.shape
    omega_k = np.fft.fft2(omega)

    ky = np.fft.fftfreq(H) * H
    kx = np.fft.fftfreq(W) * W
    KX, KY = np.meshgrid(kx, ky, indexing="xy")
    k2 = (KX**2 + KY**2).astype(np.float64)

    psi_k = np.zeros_like(omega_k)
    mask = k2 > 0
    psi_k[mask] = omega_k[mask] / (k2[mask] + eps)

    psi = np.fft.ifft2(psi_k).real.astype(np.float32)
    return psi

def velocity_from_streamfunction(psi: np.ndarray, L: float = 2*math.pi):
    H, W = psi.shape
    dx = L / W
    dy = L / H
    u = (np.roll(psi, -1, axis=0) - np.roll(psi, 1, axis=0)) / (2 * dy)
    v = -(np.roll(psi, -1, axis=1) - np.roll(psi, 1, axis=1)) / (2 * dx)
    return u.astype(np.float32), v.astype(np.float32)

def divergence(u: np.ndarray, v: np.ndarray, L: float = 2*math.pi):
    H, W = u.shape
    dx = L / W
    dy = L / H
    du_dx = (np.roll(u, -1, axis=1) - np.roll(u, 1, axis=1)) / (2 * dx)
    dv_dy = (np.roll(v, -1, axis=0) - np.roll(v, 1, axis=0)) / (2 * dy)
    return (du_dx + dv_dy).astype(np.float32)

def kinetic_energy(u: np.ndarray, v: np.ndarray) -> float:
    return float(0.5 * np.mean(u*u + v*v))


In [ ]:
# %%
from torch.cuda.amp import autocast, GradScaler

def _amp_enabled(device: torch.device) -> bool:
    return (device.type == "cuda")

def train_fm(model: nn.Module, train_loader, val_loader, epochs: int, lr: float, weight_decay: float,
             fm_t_eps: float, seed: int, out_path: str,
             log_every_batches: int = 10,
             use_amp: bool = True,
             val_every: int = 1):
    set_seed(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    amp_on = use_amp and _amp_enabled(DEVICE)
    scaler = GradScaler(enabled=amp_on)

    hist = {"train_loss": [], "val_loss": []}
    best_val = float("inf")
    t_global0 = time.time()

    for ep in range(1, epochs + 1):
        ep_t0 = time.time()

        # ---------------- train ----------------
        model.train()
        losses = []
        n_seen = 0

        for it, x1 in enumerate(train_loader):
            if it % log_every_batches == 0:
                print(f"[FM][ep {ep:03d}] train it={it}/{len(train_loader)}", flush=True)

            x1 = x1.to(DEVICE, non_blocking=True)
            B = x1.shape[0]
            n_seen += B

            x0 = torch.randn_like(x1)
            t = torch.rand((B,), device=DEVICE) * (1 - 2*fm_t_eps) + fm_t_eps
            xt = fm_sample_xt(x0, x1, t)
            target_u = (x1 - x0)

            opt.zero_grad(set_to_none=True)

            with autocast(enabled=amp_on):
                v_hat = model(xt, t)
                loss = F.mse_loss(v_hat, target_u)

            scaler.scale(loss).backward()

            # IMPORTANT: unscale before clipping
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(opt)
            scaler.update()

            losses.append(loss.item())

        train_loss = float(np.mean(losses)) if len(losses) else float("nan")

        # ---------------- val ----------------
        do_val = (val_every is not None) and (val_every >= 1) and (ep % val_every == 0)
        if do_val:
            model.eval()
            vloss = []
            with torch.no_grad():
                for it, x1 in enumerate(val_loader):
                    if it % log_every_batches == 0:
                        print(f"[FM][ep {ep:03d}] val   it={it}/{len(val_loader)}", flush=True)

                    x1 = x1.to(DEVICE, non_blocking=True)
                    B = x1.shape[0]

                    x0 = torch.randn_like(x1)
                    t = torch.rand((B,), device=DEVICE) * (1 - 2*fm_t_eps) + fm_t_eps
                    xt = fm_sample_xt(x0, x1, t)
                    target_u = (x1 - x0)

                    with autocast(enabled=amp_on):
                        v_hat = model(xt, t)
                        vloss.append(F.mse_loss(v_hat, target_u).item())

            val_loss = float(np.mean(vloss)) if len(vloss) else float("nan")

            # save best
            if val_loss < best_val:
                best_val = val_loss
                torch.save({"model": model.state_dict(), "seed": seed, "best_val": best_val}, out_path)
        else:
            val_loss = float("nan")

        hist["train_loss"].append(train_loss)
        hist["val_loss"].append(val_loss)

        ep_dt = time.time() - ep_t0
        total_dt = time.time() - t_global0
        sps = n_seen / max(ep_dt, 1e-9)

        print(f"[FM] ep {ep:03d} | train {train_loss:.4e} | val {val_loss:.4e} | "
              f"ep_dt {ep_dt:.1f}s | total {total_dt/60:.1f}m | {sps:.1f} samples/s | amp={amp_on}",
              flush=True)

    return hist


def train_ddpm(model: nn.Module, ddpm: DDPM, train_loader, val_loader,
               epochs: int, lr: float, weight_decay: float,
               seed: int, out_path: str,
               log_every_batches: int = 10,
               use_amp: bool = True,
               val_every: int = 1):
    set_seed(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    amp_on = use_amp and _amp_enabled(DEVICE)
    scaler = GradScaler(enabled=amp_on)

    hist = {"train_loss": [], "val_loss": []}
    best_val = float("inf")
    t_global0 = time.time()

    for ep in range(1, epochs + 1):
        ep_t0 = time.time()

        # ---------------- train ----------------
        model.train()
        losses = []
        n_seen = 0

        for it, x0 in enumerate(train_loader):
            if it % log_every_batches == 0:
                print(f"[DDPM][ep {ep:03d}] train it={it}/{len(train_loader)}", flush=True)

            x0 = x0.to(DEVICE, non_blocking=True)
            B = x0.shape[0]
            n_seen += B

            t_idx = torch.randint(0, ddpm.T, (B,), device=DEVICE)
            eps = torch.randn_like(x0)
            xt = ddpm.q_sample(x0, t_idx, eps)
            t = (t_idx.float() + 1) / ddpm.T

            opt.zero_grad(set_to_none=True)

            with autocast(enabled=amp_on):
                eps_hat = model(xt, t)
                loss = F.mse_loss(eps_hat, eps)

            scaler.scale(loss).backward()

            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(opt)
            scaler.update()

            losses.append(loss.item())

        train_loss = float(np.mean(losses)) if len(losses) else float("nan")

        # ---------------- val ----------------
        do_val = (val_every is not None) and (val_every >= 1) and (ep % val_every == 0)
        if do_val:
            model.eval()
            vloss = []
            with torch.no_grad():
                for it, x0 in enumerate(val_loader):
                    if it % log_every_batches == 0:
                        print(f"[DDPM][ep {ep:03d}] val   it={it}/{len(val_loader)}", flush=True)

                    x0 = x0.to(DEVICE, non_blocking=True)
                    B = x0.shape[0]

                    t_idx = torch.randint(0, ddpm.T, (B,), device=DEVICE)
                    eps = torch.randn_like(x0)
                    xt = ddpm.q_sample(x0, t_idx, eps)
                    t = (t_idx.float() + 1) / ddpm.T

                    with autocast(enabled=amp_on):
                        eps_hat = model(xt, t)
                        vloss.append(F.mse_loss(eps_hat, eps).item())

            val_loss = float(np.mean(vloss)) if len(vloss) else float("nan")

            # save best
            if val_loss < best_val:
                best_val = val_loss
                torch.save({"model": model.state_dict(), "seed": seed, "best_val": best_val}, out_path)
        else:
            val_loss = float("nan")

        hist["train_loss"].append(train_loss)
        hist["val_loss"].append(val_loss)

        ep_dt = time.time() - ep_t0
        total_dt = time.time() - t_global0
        sps = n_seen / max(ep_dt, 1e-9)

        print(f"[DDPM] ep {ep:03d} | train {train_loss:.4e} | val {val_loss:.4e} | "
              f"ep_dt {ep_dt:.1f}s | total {total_dt/60:.1f}m | {sps:.1f} samples/s | amp={amp_on}",
              flush=True)

    return hist

In [ ]:
# %%
SMOKE_TEST = False

if SMOKE_TEST:
    n_train_sm, n_val_sm, n_test_sm = 2000, 500, 500
    epochs_run = 20
    batch_size_sm = 64

    train_loader_run = DataLoader(OmegaDataset(omegas_train_z[:n_train_sm]),
                                  batch_size=batch_size_sm, shuffle=True, drop_last=True)
    val_loader_run   = DataLoader(OmegaDataset(omegas_val_z[:n_val_sm]),
                                  batch_size=batch_size_sm, shuffle=False)
    test_real = omegas_test_z[:n_test_sm]
else:
    train_loader_run, val_loader_run = train_loader, val_loader
    test_real = omegas_test_z
    epochs_run = cfg.epochs

# Global fair color scale (based on REAL set)
GLOBAL_VLIM = tuple(np.percentile(test_real, [2, 98]))
print("GLOBAL_VLIM:", GLOBAL_VLIM)

seeds = [17, 42, 123]
run_dir = os.path.join(cfg.out_dir, cfg.run_name + ("_smoke" if SMOKE_TEST else "_full"))
os.makedirs(run_dir, exist_ok=True)

all_hist = {}

for seed in seeds:
    print("\n" + "="*80)
    print("Seed:", seed)

    fm_model = UNetLite(in_ch=1, out_ch=1, base_ch=cfg.base_ch, down_levels=cfg.down_levels).to(DEVICE)
    dd_model = UNetLite(in_ch=1, out_ch=1, base_ch=cfg.base_ch, down_levels=cfg.down_levels).to(DEVICE)

    ddpm = DDPM(cfg.ddpm_T, cfg.ddpm_beta_start, cfg.ddpm_beta_end, DEVICE)

    fm_ckpt = os.path.join(run_dir, f"fm_best_seed{seed}.pt")
    dd_ckpt = os.path.join(run_dir, f"ddpm_best_seed{seed}.pt")

    hist_fm = train_fm(
        fm_model, train_loader_run, val_loader_run,
        epochs=epochs_run, lr=cfg.lr, weight_decay=cfg.weight_decay,
        fm_t_eps=cfg.fm_t_eps, seed=seed, out_path=fm_ckpt,
        val_every=5
    )
    hist_dd = train_ddpm(
        dd_model, ddpm, train_loader_run, val_loader_run,
        epochs=epochs_run, lr=cfg.lr, weight_decay=cfg.weight_decay,
        seed=seed, out_path=dd_ckpt,
        val_every=5
    )

    all_hist[seed] = {"fm": hist_fm, "ddpm": hist_dd}

    plt.figure(figsize=(6,4))
    plt.plot(hist_fm["train_loss"], label="FM train")
    plt.plot(hist_fm["val_loss"], label="FM val")
    plt.plot(hist_dd["train_loss"], label="DDPM train")
    plt.plot(hist_dd["val_loss"], label="DDPM val")
    plt.yscale("log")
    plt.xlabel("epoch"); plt.ylabel("MSE loss (log)")
    plt.title(f"Training curves (seed={seed})")
    plt.legend()
    plt.show()


In [ ]:
# %%
def plot_omega_grid(samples: np.ndarray, title: str, nrow: int = 2, ncol: int = 4, vlim=None):
    fig, axes = plt.subplots(nrow, ncol, figsize=(10, 4))
    if vlim is None:
        vmin, vmax = np.percentile(samples, [2, 98])
    else:
        vmin, vmax = vlim

    for ax, om in zip(axes.ravel(), samples[:nrow*ncol]):
        im = ax.imshow(om, origin="lower", vmin=vmin, vmax=vmax)
        ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.7)
    plt.suptitle(title)
    plt.show()

def plot_power_spectrum_heatmap(field: np.ndarray, title: str):
    Fk = np.fft.fftshift(np.fft.fft2(field))
    P = np.log(np.abs(Fk)**2 + 1e-12)
    plt.figure(figsize=(4.8,4.2))
    plt.imshow(P, origin="lower")
    plt.title(title)
    plt.xlabel("kx"); plt.ylabel("ky")
    plt.colorbar(shrink=0.8)
    plt.show()

def plot_radial_spectrum_compare(real_batch: np.ndarray, gen_batch: np.ndarray, title: str):
    def batch_logE(batch):
        Es = []
        for f in batch:
            k, E = radial_spectrum_2d(f)
            Es.append(np.log(E + 1e-12))
        L = min(map(len, Es))
        Es = np.stack([e[:L] for e in Es], axis=0)
        return np.arange(L), Es.mean(0), Es.std(0)

    k, mR, sR = batch_logE(real_batch)
    _, mG, sG = batch_logE(gen_batch)

    plt.figure(figsize=(6,4))
    plt.plot(k[1:], mR[1:], label="real")
    plt.fill_between(k[1:], (mR-sR)[1:], (mR+sR)[1:], alpha=0.2)
    plt.plot(k[1:], mG[1:], label="gen")
    plt.fill_between(k[1:], (mG-sG)[1:], (mG+sG)[1:], alpha=0.2)
    plt.xlabel("k"); plt.ylabel("log E(k)")
    plt.title(title)
    plt.legend()
    plt.show()


In [ ]:
# %%
real_batch = test_real[:8]
plot_omega_grid(real_batch, "Figure 1A: Real omega samples (standardized)", vlim=GLOBAL_VLIM)
plot_power_spectrum_heatmap(real_batch[0], "Figure 1B: log power spectrum (real sample)")
plot_radial_spectrum_compare(real_batch, real_batch, "Figure 1C: radial spectrum (real vs real)")


In [ ]:
# %%
def build_models_for_seed(seed: int):
    fm = UNetLite(in_ch=1, out_ch=1, base_ch=cfg.base_ch, down_levels=cfg.down_levels).to(DEVICE)
    dd = UNetLite(in_ch=1, out_ch=1, base_ch=cfg.base_ch, down_levels=cfg.down_levels).to(DEVICE)
    ddpm = DDPM(cfg.ddpm_T, cfg.ddpm_beta_start, cfg.ddpm_beta_end, DEVICE)

    fm_ckpt = torch.load(os.path.join(run_dir, f"fm_best_seed{seed}.pt"), map_location=DEVICE)
    dd_ckpt = torch.load(os.path.join(run_dir, f"ddpm_best_seed{seed}.pt"), map_location=DEVICE)

    fm.load_state_dict(fm_ckpt["model"])
    dd.load_state_dict(dd_ckpt["model"])
    fm.eval(); dd.eval()
    return fm, dd, ddpm


In [ ]:
# %%
@torch.no_grad()
def fm_trajectory(model, H: int, W: int, n_steps: int = 20,
                  snap_ts: List[float] = [0,0.25,0.5,0.75,1.0], method: str="euler"):
    B = 1
    x = torch.randn((B,1,H,W), device=DEVICE)
    dt = 1.0 / n_steps
    snap_steps = sorted(set(int(round(t * n_steps)) for t in snap_ts))
    snaps = {}

    def v(x, t_scalar):
        t = torch.full((B,), t_scalar, device=DEVICE)
        return model(x, t)

    for step in range(n_steps + 1):
        t = step * dt
        if step in snap_steps:
            snaps[float(t)] = x[0,0].detach().cpu().numpy()

        if step == n_steps:
            break

        if method == "euler":
            x = x + dt * v(x, t)
        elif method == "rk4":
            k1 = v(x, t)
            k2 = v(x + 0.5*dt*k1, t + 0.5*dt)
            k3 = v(x + 0.5*dt*k2, t + 0.5*dt)
            k4 = v(x + dt*k3, t + dt)
            x = x + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4)
        else:
            raise ValueError("method must be 'euler' or 'rk4'")

    return snaps

seed0 = seeds[0]
fm_model, dd_model, ddpm = build_models_for_seed(seed0)

snaps = fm_trajectory(fm_model, H=cfg.H, W=cfg.W, n_steps=20, method="euler")
ts = sorted(snaps.keys())
fields = np.stack([snaps[t] for t in ts], axis=0)

plot_omega_grid(fields, f"Figure 2A: FM omega_t snapshots (Euler, seed={seed0})",
               nrow=1, ncol=len(ts), vlim=GLOBAL_VLIM)

plt.figure(figsize=(6,4))
for t in ts:
    k, E = radial_spectrum_2d(snaps[t])
    plt.plot(k[1:], np.log(E[1:]+1e-12), label=f"t={t:.2f}")
plt.xlabel("k"); plt.ylabel("log E_t(k)")
plt.title(f"Figure 2B: FM spectrum evolution (Euler, seed={seed0})")
plt.legend(ncol=2, fontsize=8)
plt.show()

Ss = [structure_complexity(snaps[t]) for t in ts]
plt.figure(figsize=(5,3))
plt.plot(ts, Ss, marker="o")
plt.xlabel("t"); plt.ylabel("S(t)=||grad omega_t||_2")
plt.title(f"FM structure complexity over time (seed={seed0})")
plt.show()


In [ ]:
# %%
@torch.no_grad()
def ddpm_trajectory(model, ddpm: DDPM, H: int, W: int, steps: int = 50,
                    snap_fracs: List[float] = [1.0,0.75,0.5,0.25,0.0]):
    B = 1
    x = torch.randn((B,1,H,W), device=DEVICE)

    if steps >= ddpm.T:
        idxs = list(range(ddpm.T - 1, -1, -1))
    else:
        stride = (ddpm.T - 1) / (steps - 1)
        idxs = [int(round(i * stride)) for i in reversed(range(steps))]
        idxs = sorted(list(set(idxs)), reverse=True)
        if idxs[-1] != 0:
            idxs.append(0)

    desired = [int(round(frac * (ddpm.T - 1))) for frac in snap_fracs]
    present = np.array(idxs, dtype=int)

    snap_targets = {}
    for frac, d in zip(snap_fracs, desired):
        nearest = int(present[np.argmin(np.abs(present - d))])
        snap_targets[nearest] = float(nearest / (ddpm.T - 1))

    snaps = {}
    for t_idx in idxs:
        if t_idx in snap_targets:
            snaps[snap_targets[t_idx]] = x[0,0].detach().cpu().numpy()
        x = ddpm.p_sample(model, x, t_idx)

    snaps[0.0] = x[0,0].detach().cpu().numpy()
    return snaps

snaps_d = ddpm_trajectory(dd_model, ddpm, H=cfg.H, W=cfg.W, steps=50)
fracs = sorted(snaps_d.keys(), reverse=True)
fields_d = np.stack([snaps_d[f] for f in fracs], axis=0)

plot_omega_grid(fields_d, f"Figure 3A: DDPM snapshots (viz steps=50, seed={seed0})",
               nrow=1, ncol=len(fracs), vlim=GLOBAL_VLIM)

plt.figure(figsize=(6,4))
for f in fracs:
    k, E = radial_spectrum_2d(snaps_d[f])
    plt.plot(k[1:], np.log(E[1:]+1e-12), label=f"frac={f:.2f}")
plt.xlabel("k"); plt.ylabel("log E(k)")
plt.title(f"Figure 3B: DDPM spectrum recovery (viz steps=50, seed={seed0})")
plt.legend(ncol=2, fontsize=8)
plt.show()


In [ ]:
# %%
def plot_streamlines_on_omega(omega: np.ndarray, title: str, density: float = 1.2, vlim=None):
    psi = poisson_solve_streamfunction_from_vorticity(omega)
    u, v = velocity_from_streamfunction(psi)

    H, W = omega.shape
    x = np.linspace(0, 2*math.pi, W, endpoint=False)
    y = np.linspace(0, 2*math.pi, H, endpoint=False)
    X, Y = np.meshgrid(x, y, indexing="xy")

    plt.figure(figsize=(5,4))
    if vlim is None:
        vmin, vmax = np.percentile(omega, [2, 98])
    else:
        vmin, vmax = vlim
    plt.imshow(omega, origin="lower", extent=[0,2*math.pi,0,2*math.pi], vmin=vmin, vmax=vmax)
    plt.streamplot(X, Y, u, v, density=density, linewidth=0.6)
    plt.title(title)
    plt.xlabel("x"); plt.ylabel("y")
    plt.colorbar(shrink=0.8)
    plt.show()

with torch.no_grad():
    fm_samp = fm_sample_ode(fm_model, (1,1,cfg.H,cfg.W), n_steps=10, method="euler", device=DEVICE)
    dd_samp = ddpm.sample_full(dd_model, (1,1,cfg.H,cfg.W))

real0 = test_real[0]
fm0 = fm_samp[0,0].detach().cpu().numpy()
dd0 = dd_samp[0,0].detach().cpu().numpy()

plot_streamlines_on_omega(real0, "Figure 4: Real omega + streamlines", vlim=GLOBAL_VLIM)
plot_streamlines_on_omega(fm0,   "Figure 4: FM omega + streamlines",   vlim=GLOBAL_VLIM)
plot_streamlines_on_omega(dd0,   "Figure 4: DDPM omega + streamlines", vlim=GLOBAL_VLIM)

# Optional physical checks (print)
psi = poisson_solve_streamfunction_from_vorticity(dd0)
u, v = velocity_from_streamfunction(psi)
div = divergence(u, v)
print("DDPM div L2:", float(np.sqrt(np.mean(div*div))), "KE:", kinetic_energy(u, v))


In [ ]:
# %%
@torch.no_grad()
def eval_tradeoff_for_model(fm_model, dd_model, ddpm: DDPM, real_batch: np.ndarray):
    results = []

    # FM Euler 5/10/20
    for n in [5, 10, 20]:
        gen = fm_sample_ode(fm_model, (len(real_batch),1,cfg.H,cfg.W), n_steps=n, method="euler", device=DEVICE)
        gen_np = gen[:,0].detach().cpu().numpy()
        m = batch_metrics(gen_np, real_batch)
        m.update({"method":"FM-Euler", "steps": n})
        results.append(m)

    # FM RK4 5/10/20
    for n in [5, 10, 20]:
        gen = fm_sample_ode(fm_model, (len(real_batch),1,cfg.H,cfg.W), n_steps=n, method="rk4", device=DEVICE)
        gen_np = gen[:,0].detach().cpu().numpy()
        m = batch_metrics(gen_np, real_batch)
        m.update({"method":"FM-RK4", "steps": n})
        results.append(m)

    # DDPM strict: only full chain for quantitative comparison
    gen = ddpm.sample_full(dd_model, (len(real_batch),1,cfg.H,cfg.W))
    gen_np = gen[:,0].detach().cpu().numpy()
    m = batch_metrics(gen_np, real_batch)
    m.update({"method":"DDPM", "steps": ddpm.T})
    results.append(m)

    return results

def plot_tradeoff(results, metric: str, title: str):
    groups = {}
    for r in results:
        groups.setdefault(r["method"], []).append((r["steps"], r[metric]))
    plt.figure(figsize=(6,4))
    for k, pts in groups.items():
        pts = sorted(pts, key=lambda x: x[0])
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        plt.plot(xs, ys, marker="o", label=k)
    plt.xlabel("sampling steps")
    plt.ylabel(metric)
    plt.title(title)
    plt.legend()
    plt.show()

real_eval = test_real[:64]
results_seed0 = eval_tradeoff_for_model(fm_model, dd_model, ddpm, real_eval)
print(results_seed0)

plot_tradeoff(results_seed0, "SpecErr", "Figure 5: SpecErr vs steps (single seed)")
plot_tradeoff(results_seed0, "H1Err",   "Figure 5: H1Err vs steps (single seed)")


In [ ]:
# %%
import pandas as pd

real_eval = test_real[:64]

all_rows = []
for seed in seeds:
    fm_s, dd_s, ddpm_s = build_models_for_seed(seed)
    res = eval_tradeoff_for_model(fm_s, dd_s, ddpm_s, real_eval)
    for r in res:
        all_rows.append({"seed": seed, **r})

df = pd.DataFrame(all_rows)
display(df)

summary = df.groupby(["method", "steps"]).agg(
    SpecErr_mean=("SpecErr", "mean"),
    SpecErr_std=("SpecErr", "std"),
    H1Err_mean=("H1Err", "mean"),
    H1Err_std=("H1Err", "std"),
).reset_index()

display(summary)

def plot_tradeoff_with_error(summary: pd.DataFrame, metric_mean: str, metric_std: str, title: str):
    plt.figure(figsize=(6,4))
    for method in summary["method"].unique():
        sub = summary[summary["method"] == method].sort_values("steps")
        xs = sub["steps"].to_numpy()
        ys = sub[metric_mean].to_numpy()
        es = sub[metric_std].to_numpy()
        plt.plot(xs, ys, marker="o", label=method)
        plt.fill_between(xs, ys-es, ys+es, alpha=0.2)
    plt.xlabel("sampling steps")
    plt.ylabel(metric_mean.replace("_mean",""))
    plt.title(title)
    plt.legend()
    plt.show()

plot_tradeoff_with_error(summary, "SpecErr_mean", "SpecErr_std", "Figure 5: SpecErr vs steps (mean±std over seeds)")
plot_tradeoff_with_error(summary, "H1Err_mean",   "H1Err_std",   "Figure 5: H1Err vs steps (mean±std over seeds)")


In [ ]:
# %%
def moments(x: np.ndarray):
    z = x.reshape(-1).astype(np.float64)
    m = z.mean()
    v = z.var() + 1e-12
    s = ((z - m)**3).mean() / (v**1.5)
    k = ((z - m)**4).mean() / (v**2)
    return {"mean": float(m), "var": float(v), "skew": float(s), "kurt": float(k)}

def plot_hist_real_gen(real: np.ndarray, gen: np.ndarray, title: str, bins: int = 120):
    r = real.reshape(-1)
    g = gen.reshape(-1)
    lo = np.percentile(np.concatenate([r, g]), 0.5)
    hi = np.percentile(np.concatenate([r, g]), 99.5)
    bs = np.linspace(lo, hi, bins)

    plt.figure(figsize=(6,4))
    plt.hist(r, bins=bs, density=True, alpha=0.5, label="real")
    plt.hist(g, bins=bs, density=True, alpha=0.5, label="gen")
    plt.xlabel("pixel value"); plt.ylabel("density")
    plt.title(title)
    plt.legend()
    plt.show()

@torch.no_grad()
def sample_many_fm(fm_model, n: int, steps: int = 10, method: str = "euler"):
    x = fm_sample_ode(fm_model, (n,1,cfg.H,cfg.W), n_steps=steps, method=method, device=DEVICE)
    return x[:,0].detach().cpu().numpy()

@torch.no_grad()
def sample_many_ddpm_full(dd_model, ddpm: DDPM, n: int):
    x = ddpm.sample_full(dd_model, (n,1,cfg.H,cfg.W))
    return x[:,0].detach().cpu().numpy()

n_stat = 256
real_stat = test_real[:n_stat]
fm_stat = sample_many_fm(fm_model, n_stat, steps=10, method="euler")
dd_stat = sample_many_ddpm_full(dd_model, ddpm, n_stat)

print("Real moments:", moments(real_stat))
print("FM moments  :", moments(fm_stat))
print("DDPM moments:", moments(dd_stat))

plot_hist_real_gen(real_stat, fm_stat, "Pixel histogram: Real vs FM (Euler, 10 steps)")
plot_hist_real_gen(real_stat, dd_stat, "Pixel histogram: Real vs DDPM (full 100 steps)")


In [ ]:
# %%
fig_dir = os.path.join(run_dir, "figures")
os.makedirs(fig_dir, exist_ok=True)

def savefig(name: str):
    plt.savefig(os.path.join(fig_dir, name + ".png"), bbox_inches="tight")
    plt.savefig(os.path.join(fig_dir, name + ".pdf"), bbox_inches="tight")
